In [1]:
import os
import pandas as pd
import numpy as np
import boto3
from tqdm import tqdm
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
# Suppress PerformanceWarning
warnings.filterwarnings('ignore')
import sklearn.metrics as skm

try:
    import catboost as cb
except:
    ! pip install catboost

### Functions

In [2]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # download file
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)

In [3]:
# upload to s3
def upload_to_s3(str_local_path, str_bucket_key, str_bucket_name):
    # upload file
    boto3.client('s3').upload_file(str_project, str_bucket_path, str_local_path)

### Constants

In [4]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')
str_dirname_output = './output'
str_variant = 'noPTImodel7'

# indicators
dict_model_column = {
    'DQ1_1': 'Early_Pay_Delinquency_1_30_Flag',
    'DQ1_2': 'Early_Pay_Delinquency_1_60_Flag',
    'DQ1_3': 'Early_Pay_Delinquency_1_90_Flag',
    'DQ1_4': 'Early_Pay_Delinquency_1_120_Flag',
    'DQ1_5': 'Early_Pay_Delinquency_1_150_Flag',
    'DQ1_6': 'Early_Pay_Delinquency_1_180_Flag',
    'DQ1_7': 'Early_Pay_Delinquency_1_210_Flag',
    'DQ1_8': 'Early_Pay_Delinquency_1_240_Flag',
    'DQ1_9': 'Early_Pay_Delinquency_1_270_Flag',
    'DQ1_10': 'Early_Pay_Delinquency_1_300_Flag',
    'DQ1_11': 'Early_Pay_Delinquency_1_330_Flag',
    'DQ1_12': 'Early_Pay_Delinquency_1_360_Flag',
    'DQ1_13': 'Early_Pay_Delinquency_1_390_Flag',
    'DQ1_14': 'Early_Pay_Delinquency_1_420_Flag',
    'DQ1_15': 'Early_Pay_Delinquency_1_450_Flag',
    'DQ1_16': 'Early_Pay_Delinquency_1_480_Flag',
    'DQ1_17': 'Early_Pay_Delinquency_1_510_Flag',
    'DQ1_18': 'Early_Pay_Delinquency_1_540_Flag',
    'DQ1_19': 'Early_Pay_Delinquency_1_570_Flag',
    'DQ1_20': 'Early_Pay_Delinquency_1_600_Flag',
    'DQ1_21': 'Early_Pay_Delinquency_1_630_Flag',
    'DQ1_22': 'Early_Pay_Delinquency_1_660_Flag',
    'DQ1_23': 'Early_Pay_Delinquency_1_690_Flag',
    'DQ1_24': 'Early_Pay_Delinquency_1_720_Flag',
    'DQ15_1': 'Early_Pay_Delinquency_15_30_Flag',
    'DQ15_2': 'Early_Pay_Delinquency_15_60_Flag',
    'DQ15_3': 'Early_Pay_Delinquency_15_90_Flag',
    'DQ15_4': 'Early_Pay_Delinquency_15_120_Flag',
    'DQ15_5': 'Early_Pay_Delinquency_15_150_Flag',
    'DQ15_6': 'Early_Pay_Delinquency_15_180_Flag',
    'DQ15_7': 'Early_Pay_Delinquency_15_210_Flag',
    'DQ15_8': 'Early_Pay_Delinquency_15_240_Flag',
    'DQ15_9': 'Early_Pay_Delinquency_15_270_Flag',
    'DQ15_10': 'Early_Pay_Delinquency_15_300_Flag',
    'DQ15_11': 'Early_Pay_Delinquency_15_330_Flag',
    'DQ15_12': 'Early_Pay_Delinquency_15_360_Flag',
    'DQ15_13': 'Early_Pay_Delinquency_15_390_Flag',
    'DQ15_14': 'Early_Pay_Delinquency_15_420_Flag',
    'DQ15_15': 'Early_Pay_Delinquency_15_450_Flag',
    'DQ15_16': 'Early_Pay_Delinquency_15_480_Flag',
    'DQ15_17': 'Early_Pay_Delinquency_15_510_Flag',
    'DQ15_18': 'Early_Pay_Delinquency_15_540_Flag',
    'DQ15_19': 'Early_Pay_Delinquency_15_570_Flag',
    'DQ15_20': 'Early_Pay_Delinquency_15_600_Flag',
    'DQ15_21': 'Early_Pay_Delinquency_15_630_Flag',
    'DQ15_22': 'Early_Pay_Delinquency_15_660_Flag',
    'DQ15_23': 'Early_Pay_Delinquency_15_690_Flag',
    'DQ15_24': 'Early_Pay_Delinquency_15_720_Flag',
    'DQ30_1': 'Early_Pay_Delinquency_30_30_Flag',
    'DQ30_2': 'Early_Pay_Delinquency_30_60_Flag',
    'DQ30_3': 'Early_Pay_Delinquency_30_90_Flag',
    'DQ30_4': 'Early_Pay_Delinquency_30_120_Flag',
    'DQ30_5': 'Early_Pay_Delinquency_30_150_Flag',
    'DQ30_6': 'Early_Pay_Delinquency_30_180_Flag',
    'DQ30_7': 'Early_Pay_Delinquency_30_210_Flag',
    'DQ30_8': 'Early_Pay_Delinquency_30_240_Flag',
    'DQ30_9': 'Early_Pay_Delinquency_30_270_Flag',
    'DQ30_10': 'Early_Pay_Delinquency_30_300_Flag',
    'DQ30_11': 'Early_Pay_Delinquency_30_330_Flag',
    'DQ30_12': 'Early_Pay_Delinquency_30_360_Flag',
    'DQ30_13': 'Early_Pay_Delinquency_30_390_Flag',
    'DQ30_14': 'Early_Pay_Delinquency_30_420_Flag',
    'DQ30_15': 'Early_Pay_Delinquency_30_450_Flag',
    'DQ30_16': 'Early_Pay_Delinquency_30_480_Flag',
    'DQ30_17': 'Early_Pay_Delinquency_30_510_Flag',
    'DQ30_18': 'Early_Pay_Delinquency_30_540_Flag',
    'DQ30_19': 'Early_Pay_Delinquency_30_570_Flag',
    'DQ30_20': 'Early_Pay_Delinquency_30_600_Flag',
    'DQ30_21': 'Early_Pay_Delinquency_30_630_Flag',
    'DQ30_22': 'Early_Pay_Delinquency_30_660_Flag',
    'DQ30_23': 'Early_Pay_Delinquency_30_690_Flag',
    'DQ30_24': 'Early_Pay_Delinquency_30_720_Flag',
    'DQ60_3': 'Early_Pay_Delinquency_60_90_Flag',
    'DQ60_4': 'Early_Pay_Delinquency_60_120_Flag',
    'DQ60_5': 'Early_Pay_Delinquency_60_150_Flag',
    'DQ60_6': 'Early_Pay_Delinquency_60_180_Flag',
    'DQ60_7': 'Early_Pay_Delinquency_60_210_Flag',
    'DQ60_8': 'Early_Pay_Delinquency_60_240_Flag',
    'DQ60_9': 'Early_Pay_Delinquency_60_270_Flag',
    'DQ60_10': 'Early_Pay_Delinquency_60_300_Flag',
    'DQ60_11': 'Early_Pay_Delinquency_60_330_Flag',
    'DQ60_12': 'Early_Pay_Delinquency_60_360_Flag',
    'DQ60_13': 'Early_Pay_Delinquency_60_390_Flag',
    'DQ60_14': 'Early_Pay_Delinquency_60_420_Flag',
    'DQ60_15': 'Early_Pay_Delinquency_60_450_Flag',
    'DQ60_16': 'Early_Pay_Delinquency_60_480_Flag',
    'DQ60_17': 'Early_Pay_Delinquency_60_510_Flag',
    'DQ60_18': 'Early_Pay_Delinquency_60_540_Flag',
    'DQ60_19': 'Early_Pay_Delinquency_60_570_Flag',
    'DQ60_20': 'Early_Pay_Delinquency_60_600_Flag',
    'DQ60_21': 'Early_Pay_Delinquency_60_630_Flag',
    'DQ60_22': 'Early_Pay_Delinquency_60_660_Flag',
    'DQ60_23': 'Early_Pay_Delinquency_60_690_Flag',
    'DQ60_24': 'Early_Pay_Delinquency_60_720_Flag',
    'DQ90_4': 'Early_Pay_Delinquency_90_120_Flag',
    'DQ90_5': 'Early_Pay_Delinquency_90_150_Flag',
    'DQ90_6': 'Early_Pay_Delinquency_90_180_Flag',
    'DQ90_7': 'Early_Pay_Delinquency_90_210_Flag',
    'DQ90_8': 'Early_Pay_Delinquency_90_240_Flag',
    'DQ90_9': 'Early_Pay_Delinquency_90_270_Flag',
    'DQ90_10': 'Early_Pay_Delinquency_90_300_Flag',
    'DQ90_11': 'Early_Pay_Delinquency_90_330_Flag',
    'DQ90_12': 'Early_Pay_Delinquency_90_360_Flag',
    'DQ90_13': 'Early_Pay_Delinquency_90_390_Flag',
    'DQ90_14': 'Early_Pay_Delinquency_90_420_Flag',
    'DQ90_15': 'Early_Pay_Delinquency_90_450_Flag',
    'DQ90_16': 'Early_Pay_Delinquency_90_480_Flag',
    'DQ90_17': 'Early_Pay_Delinquency_90_510_Flag',
    'DQ90_18': 'Early_Pay_Delinquency_90_540_Flag',
    'DQ90_19': 'Early_Pay_Delinquency_90_570_Flag',
    'DQ90_20': 'Early_Pay_Delinquency_90_600_Flag',
    'DQ90_21': 'Early_Pay_Delinquency_90_630_Flag',
    'DQ90_22': 'Early_Pay_Delinquency_90_660_Flag',
    'DQ90_23': 'Early_Pay_Delinquency_90_690_Flag',
    'DQ90_24': 'Early_Pay_Delinquency_90_720_Flag',
}

# months (production data goes from 2021-09-27 to 2023-11-27)
list_str_year_month = [
    '2021-10',
    '2021-11',
    '2021-12',
    '2022-01',
    '2022-02',
#     '2022-03',
#     '2022-04',
#     '2022-05',
#     '2022-06',
#     '2022-07',
#     '2022-08',
#     '2022-09',
#     '2022-10',
#     '2022-11',
#     '2022-12',
#     '2023-01',
#     '2023-02',
#     '2023-03',
#     '2023-04',
#     '2023-05',
#     '2023-06',
#     '2023-07',
#     '2023-08',
#     '2023-09',
#     '2023-10',
]
# note: production targets were pulled in 2024-02
# maximum days total of the targets is 720
# thus, to ensure all accounts have been 720 days (24 months) mature, I have subsetted to the latest date being 2022-02
# we can edit this as needed

Project: 20231010-gen-xii
Task: 09_early_indicators


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Variant directory

In [6]:
try:
    os.mkdir(f'{str_dirname_output}/{str_variant}')
except:
    pass

### Get mean of target in training data

In [7]:
# load pd train data
str_filename = 'df_train_raw.gzip'
str_uri = f's3://{str_project}/02_pricing_pd/01_data_prep/03_train_valid_test_split/{str_filename}'
list_cols = [
    'uniqueid',
    'intopenbktype__app',
]
df_tmp = pd.read_parquet(str_uri, columns=list_cols)
df_tmp['uniqueid'] = df_tmp['uniqueid'].astype(int)
df_tmp.drop_duplicates(subset='uniqueid', keep='last', inplace=True)

# subset
df_tmp = df_tmp[df_tmp['intopenbktype__app'].isin([7,13])].copy()

# get targets
str_filename = 'df_monitoring_targets.csv'
str_uri = f's3://{str_project}/09_early_indicators/input/{str_filename}'
df_tmp2 = pd.read_csv(str_uri)
df_tmp2['UniqueID'] = df_tmp2['UniqueID'].astype(int)
df_tmp2.drop_duplicates(subset='UniqueID', keep='last', inplace=True)

# join
df_tmp = pd.merge(
    left=df_tmp,
    right=df_tmp2,
    left_on='uniqueid',
    right_on='UniqueID',
    how='inner'
)

# get means
dict_train_target_mean = {}
for str_col in tqdm(dict_model_column.keys()):
    # get mean
    flt_mean = df_tmp[str_col].mean()
    # assign
    dict_train_target_mean[str_col] = flt_mean

# save memory
del df_tmp

# show
dict_train_target_mean

100%|██████████| 115/115 [00:00<00:00, 10267.68it/s]


{'DQ1_1': 0.190649843191598,
 'DQ1_2': 0.3481146524688207,
 'DQ1_3': 0.4625847859382977,
 'DQ1_4': 0.5440157537743417,
 'DQ1_5': 0.6038946831011597,
 'DQ1_6': 0.646050616293487,
 'DQ1_7': 0.6787615782947998,
 'DQ1_8': 0.7050178688644154,
 'DQ1_9': 0.7275909853402378,
 'DQ1_10': 0.744219969367661,
 'DQ1_11': 0.7592079352344833,
 'DQ1_12': 0.7714244037634016,
 'DQ1_13': 0.7826197943257238,
 'DQ1_14': 0.7927576398512144,
 'DQ1_15': 0.8015462037779885,
 'DQ1_16': 0.8083655459120415,
 'DQ1_17': 0.8144555466413829,
 'DQ1_18': 0.819269199912479,
 'DQ1_19': 0.82470279337758,
 'DQ1_20': 0.8291517759463205,
 'DQ1_21': 0.8325796805484648,
 'DQ1_22': 0.836262854642258,
 'DQ1_23': 0.8401648311574648,
 'DQ1_24': 0.8433009991977245,
 'DQ15_1': 0.009809641893370287,
 'DQ15_2': 0.0353001239880388,
 'DQ15_3': 0.06735467872511122,
 'DQ15_4': 0.10072204799066443,
 'DQ15_5': 0.1345999562395157,
 'DQ15_6': 0.16727445117059295,
 'DQ15_7': 0.1976879877470644,
 'DQ15_8': 0.2233608051929108,
 'DQ15_9': 0.247830

### Load data from retro scoring

In [8]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/08_retro_scoring/06_create_df/{str_filename}'
df = pd.read_parquet(str_uri)

# convert to datetime
df['applicationdate__app'] = pd.to_datetime(df['applicationdate__app'])

# drop because they break preprocessing
list_cols = [
    'applicationdayofweek__app',
]
df.drop(list_cols, axis=1, inplace=True)

# get features 100% NaN and drop
ser_isnull = df.isnull().mean()
list_all_nan = list(ser_isnull[ser_isnull==1.0].index)
# logic
if str_variant == 'model3':
    list_all_nan = [col for col in list_all_nan if col != 'cvlst_s1__tu']
else:
    pass
df.drop(list_all_nan, axis=1, inplace=True)

# get month of application
df['year_month'] = df['applicationdate__app'].dt.strftime('%Y-%m')
# subset
df = df[df['year_month'].isin(list_str_year_month)].copy()

# subset
df = df[df['intopenbktype__app'].isin([7,13])].copy()

# show
df

,uniqueid__app_x,strcity__app,strname__app,strzipcode__app,bitapproved__app,bitsystemdecline__app,bitfunded__app,applicationmonth__app,applicationquarter__app,bigdealerid__app,...,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,year_month
0,5838462__7328771__20211124,Burlington,Kentucky,41005,True,nan,True,11,4,2836.0,...,500.0,484.52,0.348085,0.124236,0,0.955681,auto,0,2013-05-08 08:47:59.997,2021-11
10,5845296__7337081__20211201,WASHINGTON,District of Columbia,20002,True,nan,True,12,4,669.0,...,500.0,617.12,0.495385,0.092189,1,0.887657,suv,1,2006-09-20 10:23:04.350,2021-12
14,5826556__7314204__20211112,KIHEI,Hawaii,96753,True,nan,True,11,4,1765.0,...,0.0,861.12,0.501822,0.078483,0,0.856627,suv,0,2011-03-24 09:32:33.527,2021-11
16,5871520__7366737__20211217,TELL CITY,Indiana,47586,True,nan,True,12,4,2254.0,...,500.0,515.70,0.263486,0.148389,0,1.042040,auto,0,2012-05-07 09:47:46.767,2021-12
20,5856284__7350457__20211211,RICHMOND,Virginia,23223,True,1.0,True,12,4,929.0,...,0.0,806.59,0.395577,0.132841,1,1.007284,auto,1,2007-07-13 08:56:07.323,2021-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9417,0__0__20220222,CHANDLER,Arizona,85225,False,0.0,False,2,1,NaN,...,1000.0,510.48,0.349799,0.149798,0,1.122614,suv,0,2003-07-08 14:32:29.097,2022-02
9525,0__0__20220221,Indianapolis,Indiana,46237,False,0.0,False,2,1,NaN,...,1500.0,424.84,0.474754,0.073374,0,0.917676,suv,0,2013-07-18 09:37:34.627,2022-02
9702,0__0__20220222,Louisville,Kentucky,40272,False,0.0,False,2,1,NaN,...,0.0,529.30,0.436436,0.112239,0,0.923187,auto,1,2012-11-21 13:38:42.487,2022-02
10670,0__0__20220214,CYPRESS,Louisiana,71457,False,0.0,False,2,1,NaN,...,0.0,575.00,0.403836,0.123374,0,1.097798,auto,0,2012-06-18 09:31:54.393,2022-02


### Get actual targets

In [9]:
# get actual targets
str_filename = 'df_early_targets.csv'
str_uri = f's3://{str_project}/09_early_indicators/05_get_target_from_db/{str_filename}'
df_tmp = pd.read_csv(str_uri)

# join
df = pd.merge(
    left=df,
    right=df_tmp,
    on='bigAccountId',
    how='inner',
)

dict_prod_target_mean = {}
for key, val in tqdm(dict_model_column.items()):
    dict_prod_target_mean[key] = df[val].mean()

# show
df

100%|██████████| 115/115 [00:00<00:00, 16515.83it/s]


,uniqueid__app_x,strcity__app,strname__app,strzipcode__app,bitapproved__app,bitsystemdecline__app,bitfunded__app,applicationmonth__app,applicationquarter__app,bigdealerid__app,...,Early_Pay_Delinquency_90_450_Flag,Early_Pay_Delinquency_90_480_Flag,Early_Pay_Delinquency_90_510_Flag,Early_Pay_Delinquency_90_540_Flag,Early_Pay_Delinquency_90_570_Flag,Early_Pay_Delinquency_90_600_Flag,Early_Pay_Delinquency_90_630_Flag,Early_Pay_Delinquency_90_660_Flag,Early_Pay_Delinquency_90_690_Flag,Early_Pay_Delinquency_90_720_Flag
0,5838462__7328771__20211124,Burlington,Kentucky,41005,True,nan,True,11,4,2836.0,...,1,1,1,1,1,1,1,1,1,1
1,5845296__7337081__20211201,WASHINGTON,District of Columbia,20002,True,nan,True,12,4,669.0,...,0,0,0,0,0,0,0,0,0,0
2,5826556__7314204__20211112,KIHEI,Hawaii,96753,True,nan,True,11,4,1765.0,...,0,0,0,0,0,0,0,0,0,0
3,5871520__7366737__20211217,TELL CITY,Indiana,47586,True,nan,True,12,4,2254.0,...,1,1,1,1,1,1,1,1,1,1
4,5856284__7350457__20211211,RICHMOND,Virginia,23223,True,1.0,True,12,4,929.0,...,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
756,0__0__20220222,CHANDLER,Arizona,85225,False,0.0,False,2,1,NaN,...,0,0,0,0,0,0,0,0,0,0
757,0__0__20220221,Indianapolis,Indiana,46237,False,0.0,False,2,1,NaN,...,0,0,0,0,0,0,0,0,0,0
758,0__0__20220222,Louisville,Kentucky,40272,False,0.0,False,2,1,NaN,...,0,0,0,0,0,0,0,0,0,0
759,0__0__20220214,CYPRESS,Louisiana,71457,False,0.0,False,2,1,NaN,...,0,0,0,0,0,0,0,0,0,0


### Summary

In [10]:
list_dict_row = []
for key, val in tqdm(dict_model_column.items()):
    # get mean training actual
    flt_mean_train_actual = dict_train_target_mean[key]
    # get mean of production actuals
    flt_mean_prod_actual = dict_prod_target_mean[key]
    # dict_row
    dict_row = {
        'indicator': key,
        'mean_train_actual': flt_mean_train_actual,
        'mean_prod_actual': flt_mean_prod_actual,
    }
    list_dict_row.append(dict_row)
    
# make df 
df_summary = pd.DataFrame(list_dict_row)

# dates
df_summary['min_date'] = list_str_year_month[0]
df_summary['max_date'] = list_str_year_month[-1]

# get days delinquent
df_summary['days_delinquent'] = df_summary['indicator'].apply(
    lambda x: int(x.split('_')[0][2:]),
)

# get days total
df_summary['days_total'] = df_summary['indicator'].apply(
    lambda x: int(x.split('_')[1]) * 30, # 30 days per month
)

# show
df_summary

100%|██████████| 115/115 [00:00<00:00, 687101.08it/s]


,indicator,mean_train_actual,mean_prod_actual,min_date,max_date,days_delinquent,days_total
0,DQ1_1,0.190650,0.002628,2021-10,2022-02,1,30
1,DQ1_2,0.348115,0.407359,2021-10,2022-02,1,60
2,DQ1_3,0.462585,0.555848,2021-10,2022-02,1,90
3,DQ1_4,0.544016,0.668857,2021-10,2022-02,1,120
4,DQ1_5,0.603895,0.724047,2021-10,2022-02,1,150
...,...,...,...,...,...,...,...
110,DQ90_20,0.099920,0.233903,2021-10,2022-02,90,600
111,DQ90_21,0.108453,0.250986,2021-10,2022-02,90,630
112,DQ90_22,0.116330,0.268068,2021-10,2022-02,90,660
113,DQ90_23,0.125155,0.274639,2021-10,2022-02,90,690


### Save

In [11]:
str_filename = 'df_summary.csv'
str_local_path = f'{str_dirname_output}/{str_variant}/{str_filename}'
df_summary.to_csv(str_local_path, index=False)